In [2]:
import pandas as pd
import numpy as np
import torch
import json
import os
import pickle
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "distilbert-base-multilingual-cased"  # Multilingue → gère le français
MAX_LEN = 128      # Longueur max des notes (suffisant pour des notes courtes)
BATCH_SIZE = 8     # Petit batch pour GTX 660M / CPU
EPOCHS = 4         # Peu d'epochs : DistilBERT converge vite en fine-tuning
LEARNING_RATE = 2e-5

CATEGORY_TO_LABEL = {"normal": 0, "cardio": 1, "metabolic": 2, "infectious": 3}
LABEL_TO_CATEGORY = {v: k for k, v in CATEGORY_TO_LABEL.items()}
NUM_LABELS = len(CATEGORY_TO_LABEL)

print(f" Device : {device}")
print(f"Modèle de base : {MODEL_NAME}")
print(f"Paramètres : batch={BATCH_SIZE}, epochs={EPOCHS}, lr={LEARNING_RATE}")

 Device : cpu
Modèle de base : distilbert-base-multilingual-cased
Paramètres : batch=8, epochs=4, lr=2e-05


In [3]:
df = pd.read_csv("../data/synthetic/consultation_notes.csv")
df["label_encoded"] = df["category"].map(CATEGORY_TO_LABEL)

print(f" {len(df)} notes chargées")
print(f"Distribution : {df['category'].value_counts().to_dict()}")

# Augmentation des données : on duplique les classes minoritaires
# pour équilibrer le dataset (technique simple mais efficace)
max_count = df["category"].value_counts().max()
dfs_augmented = []

for category in df["category"].unique():
    df_cat = df[df["category"] == category]
    n_needed = max_count - len(df_cat)
    if n_needed > 0:
        df_oversampled = df_cat.sample(n=n_needed, replace=True, random_state=42)
        dfs_augmented.append(pd.concat([df_cat, df_oversampled]))
    else:
        dfs_augmented.append(df_cat)

df_balanced = pd.concat(dfs_augmented).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\n Dataset équilibré : {len(df_balanced)} notes")
print(f"Distribution : {df_balanced['category'].value_counts().to_dict()}")

# Split train/test stratifié
X_train, X_test, y_train, y_test = train_test_split(
    df_balanced["note"].values,
    df_balanced["label_encoded"].values,
    test_size=0.15,
    random_state=42,
    stratify=df_balanced["label_encoded"].values
)

print(f"\nTrain : {len(X_train)} | Test : {len(X_test)}")

 500 notes chargées
Distribution : {'normal': 203, 'metabolic': 124, 'infectious': 97, 'cardio': 76}

 Dataset équilibré : 812 notes
Distribution : {'metabolic': 203, 'normal': 203, 'infectious': 203, 'cardio': 203}

Train : 690 | Test : 122


In [6]:
print(f" Chargement du tokenizer {MODEL_NAME}...")
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)
print(" Tokenizer chargé")

# Test de tokenisation sur un exemple
sample = "Patient avec douleur thoracique sévère et tachycardie."
tokens = tokenizer(sample, truncation=True, max_length=MAX_LEN, padding="max_length")
print(f"\nExemple de tokenisation :")
print(f"Texte    : {sample}")
print(f"Input IDs: {tokens['input_ids'][:10]}... ({len(tokens['input_ids'])} tokens)")
print(f"Tokens   : {tokenizer.convert_ids_to_tokens(tokens['input_ids'][:10])}")

 Chargement du tokenizer distilbert-base-multilingual-cased...


 Tokenizer chargé

Exemple de tokenisation :
Texte    : Patient avec douleur thoracique sévère et tachycardie.
Input IDs: [101, 24714, 15617, 10460, 10149, 91712, 77586, 14945, 10598, 11189]... (128 tokens)
Tokens   : ['[CLS]', 'Pat', '##ient', 'avec', 'do', '##uleur', 'th', '##ora', '##ci', '##que']


In [7]:
class MedicalNotesDataset(Dataset):
    """
    Dataset PyTorch pour fine-tuning DistilBERT.
    Chaque sample : note médicale → label de catégorie.
    """
    def __init__(self, texts, labels, tokenizer, max_len: int):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "label": torch.tensor(self.labels[idx], dtype=torch.long)
        }


train_dataset = MedicalNotesDataset(X_train, y_train, tokenizer, MAX_LEN)
test_dataset = MedicalNotesDataset(X_test, y_test, tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f" Datasets créés")
print(f"Train : {len(train_dataset)} | Test : {len(test_dataset)}")
print(f"Batches train : {len(train_loader)}")

 Datasets créés
Train : 690 | Test : 122
Batches train : 87


In [8]:
print(f" Chargement de DistilBERT ({MODEL_NAME})...")

model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=LABEL_TO_CATEGORY,
    label2id=CATEGORY_TO_LABEL
)
model = model.to(device)

# Comptage des paramètres
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f" Modèle chargé")
print(f"Paramètres totaux     : {total_params:,}")
print(f"Paramètres entraînables : {trainable_params:,}")
print(f"Taille estimée        : ~{total_params * 4 / 1024**2:.0f} Mo")

 Chargement de DistilBERT (distilbert-base-multilingual-cased)...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


 Modèle chargé
Paramètres totaux     : 135,327,748
Paramètres entraînables : 135,327,748
Taille estimée        : ~516 Mo


In [10]:
# Optimizer avec weight decay pour régularisation
optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.01,
    eps=1e-8
)

# Scheduler : warmup linéaire puis décroissance
# Le warmup évite les grands pas de gradient au début du fine-tuning
total_steps = len(train_loader) * EPOCHS
warmup_steps = total_steps // 10  # 10% de warmup

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

criterion = torch.nn.CrossEntropyLoss()

print(f" Configuration entraînement :")
print(f"  Total steps    : {total_steps}")
print(f"  Warmup steps   : {warmup_steps}")
print(f"  Learning rate  : {LEARNING_RATE}")
print(f"\n  Sur CPU, chaque epoch prend ~5-15 min selon ta machine.")
print(f"   Laisse tourner et surveille la loss.")

 Configuration entraînement :
  Total steps    : 348
  Warmup steps   : 34
  Learning rate  : 2e-05

  Sur CPU, chaque epoch prend ~5-15 min selon ta machine.
   Laisse tourner et surveille la loss.


In [11]:
os.makedirs("../ml_models/text_distilbert", exist_ok=True)

train_losses, val_accs = [], []
best_val_acc = 0.0

print(f" Fine-tuning DistilBERT sur {EPOCHS} epochs\n")

for epoch in range(EPOCHS):
    print(f"─── Epoch {epoch+1}/{EPOCHS} ───")

    # ── Phase d'entraînement ──
    model.train()
    total_loss = 0.0
    correct, total = 0, 0

    for batch_idx, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits

        loss.backward()
        # Gradient clipping : évite les explosions de gradient
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        if (batch_idx + 1) % 10 == 0:
            print(f"  Batch {batch_idx+1}/{len(train_loader)} | Loss: {loss.item():.4f}")

    train_acc = correct / total
    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)

    # ── Phase de validation ──
    model.eval()
    val_correct, val_total = 0, 0
    all_probs, all_true = [], []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            probs = torch.softmax(logits, dim=-1)

            preds = logits.argmax(dim=-1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

            all_probs.extend(probs.cpu().numpy())
            all_true.extend(labels.cpu().numpy())

    val_acc = val_correct / val_total
    val_auc = roc_auc_score(
        all_true,
        np.array(all_probs),
        multi_class="ovr",
        average="macro"
    )
    val_accs.append(val_acc)

    print(f"  Train Loss: {avg_loss:.4f} | Train Acc: {train_acc:.3f}")
    print(f"  Val   Acc : {val_acc:.3f} | Val AUC  : {val_auc:.3f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        model.save_pretrained("../ml_models/text_distilbert/best_model")
        tokenizer.save_pretrained("../ml_models/text_distilbert/best_model")
        print(f"   Meilleur modèle sauvegardé (acc={val_acc:.3f})")

    print()

print(f" Fine-tuning terminé — Meilleure Val Accuracy : {best_val_acc:.3f}")

 Fine-tuning DistilBERT sur 4 epochs

─── Epoch 1/4 ───
  Batch 10/87 | Loss: 1.3676
  Batch 20/87 | Loss: 1.2628
  Batch 30/87 | Loss: 0.9980
  Batch 40/87 | Loss: 0.4956
  Batch 50/87 | Loss: 0.2234
  Batch 60/87 | Loss: 0.1048
  Batch 70/87 | Loss: 0.0533
  Batch 80/87 | Loss: 0.0361
  Train Loss: 0.6073 | Train Acc: 0.822
  Val   Acc : 1.000 | Val AUC  : 1.000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

   Meilleur modèle sauvegardé (acc=1.000)

─── Epoch 2/4 ───
  Batch 10/87 | Loss: 0.0239
  Batch 20/87 | Loss: 0.0196
  Batch 30/87 | Loss: 0.0212
  Batch 40/87 | Loss: 0.0137
  Batch 50/87 | Loss: 0.0127
  Batch 60/87 | Loss: 0.0124
  Batch 70/87 | Loss: 0.0108
  Batch 80/87 | Loss: 0.0102
  Train Loss: 0.0155 | Train Acc: 1.000
  Val   Acc : 1.000 | Val AUC  : 1.000

─── Epoch 3/4 ───
  Batch 10/87 | Loss: 0.0085
  Batch 20/87 | Loss: 0.0098
  Batch 30/87 | Loss: 0.0070
  Batch 40/87 | Loss: 0.0075
  Batch 50/87 | Loss: 0.0077
  Batch 60/87 | Loss: 0.0072
  Batch 70/87 | Loss: 0.0080
  Batch 80/87 | Loss: 0.0064
  Train Loss: 0.0079 | Train Acc: 1.000
  Val   Acc : 1.000 | Val AUC  : 1.000

─── Epoch 4/4 ───
  Batch 10/87 | Loss: 0.0058
  Batch 20/87 | Loss: 0.0083
  Batch 30/87 | Loss: 0.0078
  Batch 40/87 | Loss: 0.0055
  Batch 50/87 | Loss: 0.0055
  Batch 60/87 | Loss: 0.0055
  Batch 70/87 | Loss: 0.0064
  Batch 80/87 | Loss: 0.0060
  Train Loss: 0.0063 | Train Acc: 1.000
  Val  

In [12]:
# Chargement du meilleur modèle
best_model = DistilBertForSequenceClassification.from_pretrained(
    "../ml_models/text_distilbert/best_model"
)
best_model = best_model.to(device)
best_model.eval()

all_preds, all_true = [], []
with torch.no_grad():
    for batch in test_loader:
        outputs = best_model(
            input_ids=batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device)
        )
        preds = outputs.logits.argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_true.extend(batch["label"].numpy())

print(" Rapport de classification — DistilBERT fine-tuné :\n")
print(classification_report(
    all_true, all_preds,
    target_names=list(CATEGORY_TO_LABEL.keys())
))

# Comparaison avec TF-IDF
print("\n Comparaison des modèles :")
print(f"{'Modèle':<30} {'Accuracy'}")
print(f"{'─'*40}")

# Charge les métriques TF-IDF sauvegardées
try:
    with open("../ml_models/text/metrics.json") as f:
        tfidf_metrics = json.load(f)
    print(f"{'TF-IDF + LogReg':<30} {tfidf_metrics['accuracy']:.3f}")
except FileNotFoundError:
    print(f"{'TF-IDF + LogReg':<30} (métriques non trouvées)")

from sklearn.metrics import accuracy_score
distilbert_acc = accuracy_score(all_true, all_preds)
print(f"{'DistilBERT fine-tuné':<30} {distilbert_acc:.3f}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

 Rapport de classification — DistilBERT fine-tuné :

              precision    recall  f1-score   support

      normal       1.00      1.00      1.00        31
      cardio       1.00      1.00      1.00        30
   metabolic       1.00      1.00      1.00        31
  infectious       1.00      1.00      1.00        30

    accuracy                           1.00       122
   macro avg       1.00      1.00      1.00       122
weighted avg       1.00      1.00      1.00       122


 Comparaison des modèles :
Modèle                         Accuracy
────────────────────────────────────────
TF-IDF + LogReg                1.000
DistilBERT fine-tuné           1.000


In [13]:
"""
ONNX permet d'exécuter DistilBERT 2-3x plus vite en inférence CPU
sans avoir besoin de PyTorch au runtime.
"""
try:
    from transformers.onnx import export
    from pathlib import Path
    import subprocess

    print(" Export ONNX en cours...")

    result = subprocess.run([
        "python", "-m", "transformers.onnx",
        "--model=../ml_models/text_distilbert/best_model",
        "--feature=sequence-classification",
        "../ml_models/text_distilbert/onnx/"
    ], capture_output=True, text=True)

    if result.returncode == 0:
        print(" Export ONNX réussi")
        print("  → ml_models/text_distilbert/onnx/model.onnx")
    else:
        print(" Export ONNX échoué — utilisation PyTorch standard")
        print(result.stderr[:200])

except Exception as e:
    print(f" ONNX non disponible : {e}")
    print("Le modèle PyTorch sera utilisé directement")

 ONNX non disponible : No module named 'transformers.onnx'
Le modèle PyTorch sera utilisé directement


In [14]:
"""
Test d'intégration : vérifie que le modèle DistilBERT
fonctionne bien via l'interface du text_service.
"""
import sys
sys.path.insert(0, "../backend")

# Mise à jour du chemin dans text_service pour pointer vers DistilBERT
# On va créer un wrapper léger

def predict_with_distilbert(note: str) -> dict:
    """Wrapper d'inférence DistilBERT."""
    best_model.eval()
    best_tokenizer = DistilBertTokenizerFast.from_pretrained(
        "../ml_models/text_distilbert/best_model"
    )

    encoding = best_tokenizer(
        note,
        truncation=True,
        max_length=MAX_LEN,
        padding="max_length",
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = best_model(
            input_ids=encoding["input_ids"].to(device),
            attention_mask=encoding["attention_mask"].to(device)
        )
        probs = torch.softmax(outputs.logits, dim=-1)[0].cpu().numpy()
        pred_label = probs.argmax()

    return {
        "predicted_category": LABEL_TO_CATEGORY[pred_label],
        "confidence": float(probs[pred_label]),
        "probabilities": {LABEL_TO_CATEGORY[i]: float(p) for i, p in enumerate(probs)},
        "model_used": "distilbert_finetuned"
    }


# Tests sur quelques phrases
test_notes = [
    "Douleur thoracique sévère avec dyspnée et tachycardie à 125 bpm",
    "Glycémie à 15 mmol/L, patient diabétique déséquilibré",
    "Fièvre à 39.8°C depuis 4 jours, toux productive et frissons",
    "Consultation de routine, renouvellement ordonnance, asymptomatique"
]

print(" Test d'inférence DistilBERT :\n")
for note in test_notes:
    result = predict_with_distilbert(note)
    print(f" {note[:55]}...")
    print(f"   → {result['predicted_category'].upper()} ({result['confidence']*100:.1f}%)")
    print()

 Test d'inférence DistilBERT :

 Douleur thoracique sévère avec dyspnée et tachycardie à...
   → CARDIO (92.4%)

 Glycémie à 15 mmol/L, patient diabétique déséquilibré...
   → METABOLIC (92.1%)

 Fièvre à 39.8°C depuis 4 jours, toux productive et fris...
   → INFECTIOUS (93.4%)

 Consultation de routine, renouvellement ordonnance, asy...
   → NORMAL (86.2%)

